# Synthetic Data Generation for Drift Testing

This notebook generates synthetic variants of `training_data_with_features.zarr` for testing the drift detection pipeline.

Implemented synthetic perturbation methods:
- Mean/Variance Shift
- Covariance Shift
- Noise Injection

Two generation sections:
1. Baseline replication mode: replicate one baseline year across all years, then perturb selected years.
2. Per-year original mode: keep each year's original data, then perturb selected years.

Defaults:
- Baseline year: 2018 (customizable)
- Perturbed year indices: [2, 5] (0-based, customizable)

In [1]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

# -----------------------------
# User configuration
# -----------------------------
DATA_PATH = "training_data_with_features.zarr"
OUTPUT_ROOT = Path("synthetic_drift_data")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Default request-aligned settings
BASELINE_YEAR = 2018  # Customizable
PERTURB_YEAR_INDICES = [2, 5]  # 0-based year indices, customizable

# Band selection for mean/variance shift
# Confirm band index mapping for your dataset if needed.
RED_BAND_INDEX = 3
MEAN_SHIFT_FACTOR = 0.10  # +10%
MEAN_SHIFT_ADD_CONSTANT = 0.0  # Optional additive scalar

# Covariance shift configuration
COV_SHIFT_BAND_INDICES = [1, 2, 3, 4]  # subset to transform
COV_SHIFT_STRENGTH = 0.15  # off-diagonal coupling strength

# Noise injection configuration
NOISE_SIGMA_RATIO = 0.08  # sigma = ratio * per-band std
NOISE_PIXEL_FRACTION = 1.0  # 1.0 => all pixels; <1.0 => sampled subset

RUN_CONFIG = {
    "random_state": RANDOM_STATE,
    "data_path": DATA_PATH,
    "baseline_year": BASELINE_YEAR,
    "perturb_year_indices": PERTURB_YEAR_INDICES,
    "mean_shift": {
        "band_index": RED_BAND_INDEX,
        "factor": MEAN_SHIFT_FACTOR,
        "add_constant": MEAN_SHIFT_ADD_CONSTANT,
    },
    "covariance_shift": {
        "band_indices": COV_SHIFT_BAND_INDICES,
        "strength": COV_SHIFT_STRENGTH,
    },
    "noise_injection": {
        "sigma_ratio": NOISE_SIGMA_RATIO,
        "pixel_fraction": NOISE_PIXEL_FRACTION,
    },
}

print("Config ready.")
print(json.dumps(RUN_CONFIG, indent=2))

Config ready.
{
  "random_state": 42,
  "data_path": "training_data_with_features.zarr",
  "baseline_year": 2018,
  "perturb_year_indices": [
    2,
    5
  ],
  "mean_shift": {
    "band_index": 3,
    "factor": 0.1,
    "add_constant": 0.0
  },
  "covariance_shift": {
    "band_indices": [
      1,
      2,
      3,
      4
    ],
    "strength": 0.15
  },
  "noise_injection": {
    "sigma_ratio": 0.08,
    "pixel_fraction": 1.0
  }
}


In [2]:
# -----------------------------
# Load source dataset
# -----------------------------
load_start = time.time()
ds_source = xr.open_zarr(DATA_PATH)

if "s2_bands" not in ds_source:
    raise KeyError("Expected variable 's2_bands' in source dataset.")
if "year" not in ds_source.coords:
    raise KeyError("Expected coordinate 'year' in source dataset.")

s2_var = ds_source["s2_bands"]
if s2_var.ndim != 3:
    raise ValueError(f"Expected s2_bands with 3 dims (pixel, year, band). Got {s2_var.dims}.")

pixel_dim, year_dim, band_dim = s2_var.dims
year_values = ds_source[year_dim].values
n_pixels, n_years, n_bands = s2_var.shape

if BASELINE_YEAR not in set(int(v) for v in year_values):
    raise ValueError(f"Baseline year {BASELINE_YEAR} not found in dataset years: {year_values}.")

for idx in PERTURB_YEAR_INDICES:
    if idx < 0 or idx >= n_years:
        raise IndexError(f"Perturb year index {idx} is out of range [0, {n_years - 1}].")

baseline_idx = int(np.where(year_values == BASELINE_YEAR)[0][0])
perturb_year_values = [int(year_values[idx]) for idx in PERTURB_YEAR_INDICES]
synthetic_year_labels = [f"synthetic_{int(y)}" for y in year_values]

print(f"Dataset loaded in {time.time() - load_start:.1f}s")
print(f"s2_bands dims: {s2_var.dims}, shape: {s2_var.shape}")
print(f"Year values: {list(map(int, year_values))}")
print(f"Baseline year: {BASELINE_YEAR} (index={baseline_idx})")
print(f"Perturb indices: {PERTURB_YEAR_INDICES} -> years: {perturb_year_values}")

Dataset loaded in 0.9s
s2_bands dims: ('pixel', 'year', 's2_band'), shape: (8155205, 7, 7)
Year values: [2016, 2017, 2018, 2019, 2020, 2021, 2022]
Baseline year: 2018 (index=2)
Perturb indices: [2, 5] -> years: [2018, 2021]


In [3]:
# -----------------------------
# Reuse preparation functions from concept_drift_detection-discriminator.ipynb
# Build per-year train/val/test feature matrices BEFORE imputation
# -----------------------------
SPLIT_PATH = "data_split.npz"
PREP_SAMPLE_FRACTION = 1.0  # set <1.0 for quicker iterations

splits = np.load(SPLIT_PATH)
train_pixel_indices = splits["train_pixel_indices"]
val_pixel_indices = splits["val_pixel_indices"]
test_pixel_indices = splits["test_pixel_indices"]

if PREP_SAMPLE_FRACTION < 1.0:
    train_idx = np.random.choice(
        train_pixel_indices,
        size=int(len(train_pixel_indices) * PREP_SAMPLE_FRACTION),
        replace=False,
    )
    val_idx = np.random.choice(
        val_pixel_indices,
        size=int(len(val_pixel_indices) * PREP_SAMPLE_FRACTION),
        replace=False,
    )
    test_idx = np.random.choice(
        test_pixel_indices,
        size=int(len(test_pixel_indices) * PREP_SAMPLE_FRACTION),
        replace=False,
    )
else:
    train_idx = train_pixel_indices
    val_idx = val_pixel_indices
    test_idx = test_pixel_indices

ds_train = ds_source.isel(pixel=train_idx)
ds_val = ds_source.isel(pixel=val_idx)
ds_test = ds_source.isel(pixel=test_idx)


def prepare_features_for_year(ds_subset, year_idx, impute_nans=True, require_finite=True):
    """Same base logic as discriminator notebook, with flags for pre-imputation extraction."""
    if year_idx == 0:
        return None, None

    valid_mask = ds_subset.disturbances[:, year_idx].values != 255

    s2_data = ds_subset.s2_bands.values  # shape: (n_pixels, n_years, n_bands)
    s2_t = s2_data[:, year_idx, :].copy()  # shape: (n_pixels, n_bands)

    if impute_nans:
        nan_mask = np.isnan(s2_t)
        if np.any(nan_mask):
            band_means = np.nanmean(s2_data, axis=1)  # shape: (n_pixels, n_bands)
            s2_t[nan_mask] = band_means[nan_mask]

    X = s2_t
    y = ds_subset.disturbances[:, year_idx].values

    keep_mask = valid_mask & np.isin(y, [0, 1])
    if require_finite:
        keep_mask = keep_mask & np.all(np.isfinite(X), axis=1)

    X = X[keep_mask]
    y = y[keep_mask]

    if len(X) == 0:
        return None, None

    return X, y


def build_pair_dataset_cached(X_prev, y_prev, X_curr, y_curr):
    """Unchanged pair-builder from discriminator notebook."""
    if X_prev is None or X_curr is None:
        return None, None

    if X_prev.shape[1] != X_curr.shape[1]:
        return None, None

    X_pair = np.vstack([X_prev, X_curr])
    y_pair = np.concatenate([
        np.zeros(len(y_prev), dtype=np.uint8),
        np.ones(len(y_curr), dtype=np.uint8),
    ])

    return X_pair, y_pair


def build_per_year_feature_matrices(ds_subset, split_name, impute_nans=False, require_finite=False):
    records = []
    features = {}

    for year_idx in range(n_years):
        year_val = int(year_values[year_idx])
        X_year, y_year = prepare_features_for_year(
            ds_subset,
            year_idx,
            impute_nans=impute_nans,
            require_finite=require_finite,
        )
        features[year_idx] = (X_year, y_year)

        if X_year is None:
            records.append(
                {
                    "split": split_name,
                    "year_index": year_idx,
                    "year": year_val,
                    "n_samples": 0,
                    "n_features": None,
                    "nan_ratio": None,
                }
            )
        else:
            nan_ratio = float(np.isnan(X_year).mean()) if X_year.size > 0 else 0.0
            records.append(
                {
                    "split": split_name,
                    "year_index": year_idx,
                    "year": year_val,
                    "n_samples": int(len(y_year)),
                    "n_features": int(X_year.shape[1]),
                    "nan_ratio": nan_ratio,
                }
            )

    return features, records


# Build matrices BEFORE imputation and WITHOUT finite filtering
train_features_pre_impute, train_records = build_per_year_feature_matrices(
    ds_train,
    split_name="train",
    impute_nans=False,
    require_finite=False,
)
val_features_pre_impute, val_records = build_per_year_feature_matrices(
    ds_val,
    split_name="val",
    impute_nans=False,
    require_finite=False,
)
test_features_pre_impute, test_records = build_per_year_feature_matrices(
    ds_test,
    split_name="test",
    impute_nans=False,
    require_finite=False,
)

per_year_features_pre_impute = {
    "train": train_features_pre_impute,
    "val": val_features_pre_impute,
    "test": test_features_pre_impute,
}

prep_summary_df = pd.DataFrame(train_records + val_records + test_records)
print("Built per-year pre-imputation feature matrices for train/val/test.")
display(prep_summary_df)

Built per-year pre-imputation feature matrices for train/val/test.


,split,year_index,year,n_samples,n_features,nan_ratio
0,train,0,2016,0,NaN,NaN
1,train,1,2017,5597049,7.0,0.001172
2,train,2,2018,5597049,7.0,0.001116
3,train,3,2019,5597049,7.0,0.001144
4,train,4,2020,5597049,7.0,0.000993
5,train,5,2021,5597049,7.0,0.001111
6,train,6,2022,5597049,7.0,0.000432
7,val,0,2016,0,NaN,NaN
8,val,1,2017,1273336,7.0,0.001040
9,val,2,2018,1273336,7.0,0.001015


## Reused discriminator preparation functions (pre-imputation matrices)

This section reuses the same feature-preparation logic as the drift discriminator notebook and builds per-year train/val/test feature matrices **before imputation**.

- `impute_nans=False`
- `require_finite=False`

These matrices are intended for synthetic perturbation experiments in feature space.

In [4]:
# -----------------------------
# Perturbation utilities
# -----------------------------
def _safe_copy_s2(ds):
    return np.array(ds["s2_bands"].values, copy=True)


def _build_cov_transform_matrix(size, strength):
    # Near-identity matrix with controlled off-diagonal coupling.
    mat = np.eye(size, dtype=np.float64)
    if size > 1 and strength != 0:
        off_diag = strength / (size - 1)
        mat += off_diag * (np.ones((size, size), dtype=np.float64) - np.eye(size, dtype=np.float64))
    return mat


def apply_mean_variance_shift(s2_arr, target_year_indices, band_index, factor=0.10, add_constant=0.0):
    out = np.array(s2_arr, copy=True)
    if band_index < 0 or band_index >= out.shape[2]:
        raise IndexError(f"band_index {band_index} out of range for {out.shape[2]} bands")

    for year_idx in target_year_indices:
        band_slice = out[:, year_idx, band_index]
        shifted = band_slice * (1.0 + factor) + add_constant
        out[:, year_idx, band_index] = shifted
    return out


def apply_covariance_shift(s2_arr, target_year_indices, band_indices, strength=0.15):
    out = np.array(s2_arr, copy=True)
    band_indices = np.array(band_indices, dtype=int)

    if np.any(band_indices < 0) or np.any(band_indices >= out.shape[2]):
        raise IndexError("One or more covariance-shift band indices are out of range")

    transform = _build_cov_transform_matrix(len(band_indices), strength)
    for year_idx in target_year_indices:
        subset = out[:, year_idx, :][:, band_indices]
        transformed = subset @ transform
        out[:, year_idx, :][:, band_indices] = transformed
    return out


def apply_noise_injection(
    s2_arr,
    target_year_indices,
    sigma_ratio=0.08,
    pixel_fraction=1.0,
    random_gen=None,
):
    out = np.array(s2_arr, copy=True)
    rng_local = random_gen if random_gen is not None else np.random.default_rng(42)

    n_pix = out.shape[0]
    if pixel_fraction <= 0 or pixel_fraction > 1.0:
        raise ValueError("pixel_fraction must be in (0, 1]")

    if pixel_fraction < 1.0:
        chosen_n = max(1, int(round(n_pix * pixel_fraction)))
        pixel_idx = rng_local.choice(n_pix, size=chosen_n, replace=False)
    else:
        pixel_idx = np.arange(n_pix)

    for year_idx in target_year_indices:
        year_block = out[pixel_idx, year_idx, :]
        band_std = np.nanstd(year_block, axis=0)
        band_std = np.where(np.isfinite(band_std), band_std, 0.0)
        sigma = sigma_ratio * np.maximum(band_std, 1e-8)
        noise = rng_local.normal(loc=0.0, scale=sigma, size=year_block.shape)
        out[pixel_idx, year_idx, :] = year_block + noise
    return out


def build_synthetic_dataset(ds_source, s2_new, synthetic_year_labels_list):
    if s2_new.shape != ds_source["s2_bands"].shape:
        raise ValueError(
            f"Shape mismatch: new={s2_new.shape}, source={ds_source['s2_bands'].shape}"
        )

    ds_new = ds_source.copy(deep=True)
    ds_new["s2_bands"] = xr.DataArray(
        s2_new,
        dims=ds_source["s2_bands"].dims,
        coords=ds_source["s2_bands"].coords,
        attrs=ds_source["s2_bands"].attrs,
    )

    ds_new = ds_new.assign_coords(
        synthetic_year_label=(year_dim, np.array(synthetic_year_labels_list, dtype=object))
    )
    return ds_new


def validate_synthetic_dataset(ds_new, ds_ref):
    if ds_new["s2_bands"].shape != ds_ref["s2_bands"].shape:
        raise AssertionError("s2_bands shape changed unexpectedly")
    if list(ds_new[year_dim].values) != list(ds_ref[year_dim].values):
        raise AssertionError("year coordinate changed unexpectedly")
    if not np.all(np.isfinite(ds_new["s2_bands"].values)):
        raise AssertionError("Synthetic s2_bands contains non-finite values")


def save_synthetic_artifacts(ds_synth, metadata, output_dir, artifact_stem):
    output_dir.mkdir(parents=True, exist_ok=True)

    zarr_path = output_dir / f"{artifact_stem}.zarr"
    json_path = output_dir / f"{artifact_stem}.metadata.json"

    if zarr_path.exists():
        import shutil
        shutil.rmtree(zarr_path)

    ds_synth.to_zarr(zarr_path, mode="w")

    with open(json_path, "w", encoding="utf-8") as handle:
        json.dump(metadata, handle, indent=2)

    return zarr_path, json_path


def years_suffix(years):
    return "-".join(str(y) for y in years)


def changed_year_mask(s2_ref, s2_new):
    # True per year if any pixel/band changed.
    return np.any(np.abs(s2_new - s2_ref) > 0.0, axis=(0, 2))

In [5]:
# Validation override: allow pre-existing non-finite values from source,
# but fail if perturbation introduces new non-finite positions.
def validate_synthetic_dataset(ds_new, ds_ref):
    if ds_new["s2_bands"].shape != ds_ref["s2_bands"].shape:
        raise AssertionError("s2_bands shape changed unexpectedly")
    if list(ds_new[year_dim].values) != list(ds_ref[year_dim].values):
        raise AssertionError("year coordinate changed unexpectedly")

    ref_vals = ds_ref["s2_bands"].values
    new_vals = ds_new["s2_bands"].values

    ref_finite = np.isfinite(ref_vals)
    new_finite = np.isfinite(new_vals)

    # New synthetic data must not create additional non-finite values where source was finite.
    introduced_non_finite = ref_finite & (~new_finite)
    if np.any(introduced_non_finite):
        raise AssertionError("Synthetic s2_bands introduced new non-finite values")

In [7]:
# -----------------------------
# Feature-space synthetic generation utilities
# -----------------------------
FEATURE_OUTPUT_ROOT = OUTPUT_ROOT / "synthetic_feature_matrices"
FEATURE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


def _copy_feature_payload(payload):
    copied = {}
    for year_idx, value in payload.items():
        if value is None:
            copied[year_idx] = (None, None)
            continue
        X_year, y_year = value
        copied[year_idx] = (
            None if X_year is None else np.array(X_year, copy=True),
            None if y_year is None else np.array(y_year, copy=True),
        )
    return copied


def _baseline_feature_payload(payload, baseline_year_idx):
    copied = _copy_feature_payload(payload)
    baseline_X, baseline_y = copied[baseline_year_idx]

    if baseline_X is None or baseline_y is None:
        raise ValueError("Baseline year features are empty; cannot build baseline-replication payload.")

    for year_idx in copied:
        if year_idx == 0:
            continue
        copied[year_idx] = (np.array(baseline_X, copy=True), np.array(baseline_y, copy=True))
    return copied


def perturb_feature_matrix_mean_shift(X_year, band_index, factor=0.10, add_constant=0.0):
    X_new = np.array(X_year, copy=True)
    X_new[:, band_index] = X_new[:, band_index] * (1.0 + factor) + add_constant
    return X_new


def perturb_feature_matrix_covariance(X_year, band_indices, strength=0.15):
    X_new = np.array(X_year, copy=True)
    band_indices = np.array(band_indices, dtype=int)
    transform = _build_cov_transform_matrix(len(band_indices), strength)
    subset = X_new[:, band_indices]
    row_ok = np.all(np.isfinite(subset), axis=1)
    if np.any(row_ok):
        X_new[row_ok[:, None] if False else row_ok, :]
        subset_new = np.array(subset, copy=True)
        subset_new[row_ok] = subset[row_ok] @ transform
        X_new[:, band_indices] = subset_new
    return X_new


def perturb_feature_matrix_noise(X_year, sigma_ratio=0.08, pixel_fraction=1.0, random_gen=None):
    X_new = np.array(X_year, copy=True)
    rng_local = random_gen if random_gen is not None else np.random.default_rng(42)

    n_rows = X_new.shape[0]
    if pixel_fraction < 1.0:
        n_sel = max(1, int(round(n_rows * pixel_fraction)))
        row_idx = rng_local.choice(n_rows, size=n_sel, replace=False)
    else:
        row_idx = np.arange(n_rows)

    subset = X_new[row_idx]
    band_std = np.nanstd(subset, axis=0)
    band_std = np.where(np.isfinite(band_std), band_std, 0.0)
    sigma = sigma_ratio * np.maximum(band_std, 1e-8)
    noise = rng_local.normal(0.0, sigma, size=subset.shape)
    X_new[row_idx] = subset + noise
    return X_new


def apply_method_to_feature_payload(payload, method_name):
    copied = _copy_feature_payload(payload)

    for year_idx in PERTURB_YEAR_INDICES:
        X_year, y_year = copied.get(year_idx, (None, None))
        if X_year is None:
            continue

        if method_name == "mean_variance_shift":
            X_new = perturb_feature_matrix_mean_shift(
                X_year,
                band_index=RED_BAND_INDEX,
                factor=MEAN_SHIFT_FACTOR,
                add_constant=MEAN_SHIFT_ADD_CONSTANT,
            )
        elif method_name == "covariance_shift":
            X_new = perturb_feature_matrix_covariance(
                X_year,
                band_indices=COV_SHIFT_BAND_INDICES,
                strength=COV_SHIFT_STRENGTH,
            )
        elif method_name == "noise_injection":
            X_new = perturb_feature_matrix_noise(
                X_year,
                sigma_ratio=NOISE_SIGMA_RATIO,
                pixel_fraction=NOISE_PIXEL_FRACTION,
                random_gen=rng,
            )
        else:
            raise ValueError(f"Unknown method: {method_name}")

        copied[year_idx] = (X_new, y_year)

    return copied


def save_feature_payload(payload_by_split, metadata, output_dir, artifact_stem):
    output_dir.mkdir(parents=True, exist_ok=True)
    npz_path = output_dir / f"{artifact_stem}.npz"
    json_path = output_dir / f"{artifact_stem}.metadata.json"

    arrays_to_save = {}
    for split_name, payload in payload_by_split.items():
        for year_idx, value in payload.items():
            X_year, y_year = value
            if X_year is None:
                continue
            arrays_to_save[f"{split_name}_X_year_{year_idx}"] = X_year
            arrays_to_save[f"{split_name}_y_year_{year_idx}"] = y_year

    np.savez_compressed(npz_path, **arrays_to_save)
    with open(json_path, "w", encoding="utf-8") as handle:
        json.dump(metadata, handle, indent=2)

    return npz_path, json_path


def summarize_payload(payload_by_split, mode_name, method_name, baseline_year_value):
    rows = []
    for split_name, payload in payload_by_split.items():
        for year_idx, value in payload.items():
            X_year, y_year = value
            if X_year is None:
                continue
            rows.append(
                {
                    "mode": mode_name,
                    "method": method_name,
                    "split": split_name,
                    "baseline_year": baseline_year_value,
                    "year_index": year_idx,
                    "year": int(year_values[year_idx]),
                    "n_samples": int(X_year.shape[0]),
                    "n_features": int(X_year.shape[1]),
                    "nan_ratio": float(np.isnan(X_year).mean()) if X_year.size > 0 else 0.0,
                }
            )
    return rows

In [8]:
# -----------------------------
# Section A: baseline replication in feature space, then perturb selected years
# -----------------------------
section_a_dir = FEATURE_OUTPUT_ROOT / "section_a_baseline_replication"
section_a_dir.mkdir(parents=True, exist_ok=True)

feature_artifact_records = []
feature_summary_rows = []
methods = ["mean_variance_shift", "covariance_shift", "noise_injection"]

section_a_base = {
    "train": _baseline_feature_payload(per_year_features_pre_impute["train"], baseline_idx),
    "val": _baseline_feature_payload(per_year_features_pre_impute["val"], baseline_idx),
    "test": _baseline_feature_payload(per_year_features_pre_impute["test"], baseline_idx),
}

for method_name in methods:
    payload_by_split = {
        split_name: apply_method_to_feature_payload(split_payload, method_name)
        for split_name, split_payload in section_a_base.items()
    }

    artifact_stem = (
        f"synthetic_features_mode-baseline_replication"
        f"_method-{method_name}"
        f"_baseline-{BASELINE_YEAR}"
        f"_perturbed-{years_suffix(perturb_year_values)}"
    )

    metadata = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "source_dataset": DATA_PATH,
        "source_feature_mode": "pre_imputation",
        "mode": "baseline_replication",
        "method": method_name,
        "baseline_year": BASELINE_YEAR,
        "perturb_year_indices": PERTURB_YEAR_INDICES,
        "perturb_year_values": perturb_year_values,
        "config": RUN_CONFIG,
    }

    npz_path, json_path = save_feature_payload(payload_by_split, metadata, section_a_dir, artifact_stem)
    feature_artifact_records.append(
        {
            "section": "A",
            "mode": "baseline_replication",
            "method": method_name,
            "baseline_year": BASELINE_YEAR,
            "perturb_year_indices": PERTURB_YEAR_INDICES,
            "perturb_year_values": perturb_year_values,
            "artifact_path": str(npz_path),
            "metadata_path": str(json_path),
        }
    )
    feature_summary_rows.extend(summarize_payload(payload_by_split, "baseline_replication", method_name, BASELINE_YEAR))

print(f"Section A complete. Saved {len(methods)} synthetic feature datasets to: {section_a_dir}")

Section A complete. Saved 3 synthetic feature datasets to: synthetic_drift_data\synthetic_feature_matrices\section_a_baseline_replication


In [9]:
# -----------------------------
# Section B: per-year original feature matrices, perturb selected years
# -----------------------------
section_b_dir = FEATURE_OUTPUT_ROOT / "section_b_per_year_original"
section_b_dir.mkdir(parents=True, exist_ok=True)

section_b_base = {
    split_name: _copy_feature_payload(split_payload)
    for split_name, split_payload in per_year_features_pre_impute.items()
}

for method_name in methods:
    payload_by_split = {
        split_name: apply_method_to_feature_payload(split_payload, method_name)
        for split_name, split_payload in section_b_base.items()
    }

    artifact_stem = (
        f"synthetic_features_mode-per_year_original"
        f"_method-{method_name}"
        f"_perturbed-{years_suffix(perturb_year_values)}"
    )

    metadata = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "source_dataset": DATA_PATH,
        "source_feature_mode": "pre_imputation",
        "mode": "per_year_original",
        "method": method_name,
        "baseline_year": None,
        "perturb_year_indices": PERTURB_YEAR_INDICES,
        "perturb_year_values": perturb_year_values,
        "config": RUN_CONFIG,
    }

    npz_path, json_path = save_feature_payload(payload_by_split, metadata, section_b_dir, artifact_stem)
    feature_artifact_records.append(
        {
            "section": "B",
            "mode": "per_year_original",
            "method": method_name,
            "baseline_year": None,
            "perturb_year_indices": PERTURB_YEAR_INDICES,
            "perturb_year_values": perturb_year_values,
            "artifact_path": str(npz_path),
            "metadata_path": str(json_path),
        }
    )
    feature_summary_rows.extend(summarize_payload(payload_by_split, "per_year_original", method_name, None))

print(f"Section B complete. Saved {len(methods)} synthetic feature datasets to: {section_b_dir}")

MemoryError: Unable to allocate 299. MiB for an array with shape (5597049, 7) and data type float64

In [10]:
# -----------------------------
# Manifest + sanity summary for feature-space artifacts
# -----------------------------
manifest_df = pd.DataFrame(feature_artifact_records)
manifest_df = manifest_df.sort_values(["section", "method"]).reset_index(drop=True)

manifest_path = FEATURE_OUTPUT_ROOT / "synthetic_feature_generation_manifest.csv"
manifest_df.to_csv(manifest_path, index=False)

manifest_json_path = FEATURE_OUTPUT_ROOT / "synthetic_feature_generation_manifest.json"
with open(manifest_json_path, "w", encoding="utf-8") as handle:
    json.dump(
        {
            "generated_at_utc": datetime.now(timezone.utc).isoformat(),
            "source_dataset": DATA_PATH,
            "source_feature_mode": "pre_imputation",
            "baseline_year": BASELINE_YEAR,
            "perturb_year_indices": PERTURB_YEAR_INDICES,
            "perturb_year_values": perturb_year_values,
            "rows": manifest_df.to_dict(orient="records"),
        },
        handle,
        indent=2,
    )

feature_summary_df = pd.DataFrame(feature_summary_rows)

print(f"Saved manifest CSV:  {manifest_path}")
print(f"Saved manifest JSON: {manifest_json_path}")
print("\nGenerated feature artifacts:")
display(manifest_df)
print("\nFeature summary:")
display(feature_summary_df)

Saved manifest CSV:  synthetic_drift_data\synthetic_feature_matrices\synthetic_feature_generation_manifest.csv
Saved manifest JSON: synthetic_drift_data\synthetic_feature_matrices\synthetic_feature_generation_manifest.json

Generated feature artifacts:


,section,mode,method,baseline_year,perturb_year_indices,perturb_year_values,artifact_path,metadata_path
0,A,baseline_replication,covariance_shift,2018,"[2, 5]","[2018, 2021]",synthetic_drift_data\synthetic_feature_matrice...,synthetic_drift_data\synthetic_feature_matrice...
1,A,baseline_replication,mean_variance_shift,2018,"[2, 5]","[2018, 2021]",synthetic_drift_data\synthetic_feature_matrice...,synthetic_drift_data\synthetic_feature_matrice...
2,A,baseline_replication,noise_injection,2018,"[2, 5]","[2018, 2021]",synthetic_drift_data\synthetic_feature_matrice...,synthetic_drift_data\synthetic_feature_matrice...



Feature summary:


,mode,method,split,baseline_year,year_index,year,n_samples,n_features,nan_ratio
0,baseline_replication,mean_variance_shift,train,2018,1,2017,5597049,7,0.001116
1,baseline_replication,mean_variance_shift,train,2018,2,2018,5597049,7,0.001116
2,baseline_replication,mean_variance_shift,train,2018,3,2019,5597049,7,0.001116
3,baseline_replication,mean_variance_shift,train,2018,4,2020,5597049,7,0.001116
4,baseline_replication,mean_variance_shift,train,2018,5,2021,5597049,7,0.001116
5,baseline_replication,mean_variance_shift,train,2018,6,2022,5597049,7,0.001116
6,baseline_replication,mean_variance_shift,val,2018,1,2017,1273336,7,0.001015
7,baseline_replication,mean_variance_shift,val,2018,2,2018,1273336,7,0.001015
8,baseline_replication,mean_variance_shift,val,2018,3,2019,1273336,7,0.001015
9,baseline_replication,mean_variance_shift,val,2018,4,2020,1273336,7,0.001015


## Memory-safe generation override

This section re-runs synthetic generation with:
- NaN-safe covariance shift
- sequential per-method processing to avoid large memory spikes

In [ ]:
# -----------------------------
# Memory-safe overrides
# -----------------------------
def apply_covariance_shift(s2_arr, target_year_indices, band_indices, strength=0.15):
    out = np.array(s2_arr, copy=True)
    band_indices = np.array(band_indices, dtype=int)

    if np.any(band_indices < 0) or np.any(band_indices >= out.shape[2]):
        raise IndexError("One or more covariance-shift band indices are out of range")

    transform = _build_cov_transform_matrix(len(band_indices), strength)

    for year_idx in target_year_indices:
        subset = out[:, year_idx, :][:, band_indices]
        transformed = np.array(subset, copy=True)

        # Transform only fully finite rows; keep non-finite rows unchanged.
        row_ok = np.all(np.isfinite(subset), axis=1)
        if np.any(row_ok):
            transformed[row_ok] = subset[row_ok] @ transform

        out[:, year_idx, :][:, band_indices] = transformed

    return out


def changed_year_mask(s2_ref, s2_new):
    ref_finite = np.isfinite(s2_ref)
    new_finite = np.isfinite(s2_new)

    finite_pattern_changed = np.any(ref_finite != new_finite, axis=(0, 2))
    numeric_diff = np.nan_to_num(s2_new, nan=0.0) - np.nan_to_num(s2_ref, nan=0.0)
    values_changed = np.any(np.abs(numeric_diff) > 0.0, axis=(0, 2))

    return finite_pattern_changed | values_changed


def generate_section_artifacts(mode_name, base_s2, output_dir, baseline_year_for_meta=None):
    output_dir.mkdir(parents=True, exist_ok=True)

    method_specs = [
        ("mean_variance_shift", lambda arr: apply_mean_variance_shift(
            arr,
            target_year_indices=PERTURB_YEAR_INDICES,
            band_index=RED_BAND_INDEX,
            factor=MEAN_SHIFT_FACTOR,
            add_constant=MEAN_SHIFT_ADD_CONSTANT,
        )),
        ("covariance_shift", lambda arr: apply_covariance_shift(
            arr,
            target_year_indices=PERTURB_YEAR_INDICES,
            band_indices=COV_SHIFT_BAND_INDICES,
            strength=COV_SHIFT_STRENGTH,
        )),
        ("noise_injection", lambda arr: apply_noise_injection(
            arr,
            target_year_indices=PERTURB_YEAR_INDICES,
            sigma_ratio=NOISE_SIGMA_RATIO,
            pixel_fraction=NOISE_PIXEL_FRACTION,
            random_gen=rng,
        )),
    ]

    local_records = []

    for method_name, method_fn in method_specs:
        s2_method = method_fn(base_s2)
        ds_synth = build_synthetic_dataset(ds_source, s2_method, synthetic_year_labels)
        validate_synthetic_dataset(ds_synth, ds_source)

        if mode_name == "baseline_replication":
            artifact_stem = (
                f"synthetic_mode-baseline_replication"
                f"_method-{method_name}"
                f"_baseline-{BASELINE_YEAR}"
                f"_perturbed-{years_suffix(perturb_year_values)}"
            )
        else:
            artifact_stem = (
                f"synthetic_mode-per_year_original"
                f"_method-{method_name}"
                f"_perturbed-{years_suffix(perturb_year_values)}"
            )

        metadata = {
            "generated_at_utc": datetime.now(timezone.utc).isoformat(),
            "source_dataset": DATA_PATH,
            "mode": mode_name,
            "method": method_name,
            "baseline_year": baseline_year_for_meta,
            "perturb_year_indices": PERTURB_YEAR_INDICES,
            "perturb_year_values": perturb_year_values,
            "synthetic_year_labels": synthetic_year_labels,
            "config": RUN_CONFIG,
        }

        zarr_path, json_path = save_synthetic_artifacts(ds_synth, metadata, output_dir, artifact_stem)

        changed_mask = changed_year_mask(base_s2, s2_method)
        changed_years = [int(year_values[i]) for i, flag in enumerate(changed_mask) if flag]

        local_records.append(
            {
                "mode": mode_name,
                "method": method_name,
                "baseline_year": baseline_year_for_meta,
                "perturb_year_indices": PERTURB_YEAR_INDICES,
                "perturb_year_values": perturb_year_values,
                "changed_years": changed_years,
                "zarr_path": str(zarr_path),
                "metadata_path": str(json_path),
            }
        )

        del ds_synth
        del s2_method

    return local_records

In [ ]:
# -----------------------------
# Deprecated raw-array execution cell
# -----------------------------
# This path attempts to materialize the full s2 cube in memory and can OOM.
# Use the feature-space synthetic workflow instead:
# - Cell 4: build pre-imputation per-year feature matrices
# - Cell 9: feature-space Section A generation
# - Cell 10: feature-space Section B generation
# - Cell 11: feature-space manifest and summary
raise RuntimeError(
    "Deprecated cell: raw-array generation is disabled to avoid MemoryError. "
    "Run Cells 4, 9, 10, and 11 instead."
)

MemoryError: 

## Scalable chunked pipeline (recommended for full dataset)

Use this section for large-scale generation without loading all pixels into RAM.
It produces the same six artifacts (3 methods x 2 modes) with zarr outputs and metadata.

In [ ]:
import shutil
import dask.array as da

# Chunking strategy: keep year and band whole, chunk by pixel.
PIXEL_CHUNK = 250_000

s2_lazy = ds_source["s2_bands"].chunk({pixel_dim: PIXEL_CHUNK, year_dim: -1, band_dim: -1})


def _year_mask_values(ny, target_indices):
    mask = np.zeros(ny, dtype=bool)
    mask[target_indices] = True
    return mask


def _mean_shift_block(block, year_mask, band_index, factor, add_constant):
    out = block.copy()
    target_years = np.where(year_mask)[0]
    for yi in target_years:
        out[:, yi, band_index] = out[:, yi, band_index] * (1.0 + factor) + add_constant
    return out


def _cov_shift_block(block, year_mask, band_indices, strength):
    out = block.copy()
    band_indices = np.array(band_indices, dtype=int)
    transform = _build_cov_transform_matrix(len(band_indices), strength)

    target_years = np.where(year_mask)[0]
    for yi in target_years:
        subset = out[:, yi, :][:, band_indices]
        row_ok = np.all(np.isfinite(subset), axis=1)
        if np.any(row_ok):
            subset_new = subset.copy()
            subset_new[row_ok] = subset[row_ok] @ transform
            out[:, yi, :][:, band_indices] = subset_new
    return out


def _noise_block(block, year_mask, sigma_ratio, pixel_fraction, random_state, block_info=None):
    out = block.copy()

    chunk_loc = 0
    if block_info is not None and None in block_info:
        chunk_loc = int(block_info[None]["chunk-location"][0])
    rng_local = np.random.default_rng(random_state + chunk_loc)

    n_pix = out.shape[0]
    if pixel_fraction < 1.0:
        n_sel = max(1, int(round(n_pix * pixel_fraction)))
        pix_idx = rng_local.choice(n_pix, size=n_sel, replace=False)
    else:
        pix_idx = np.arange(n_pix)

    target_years = np.where(year_mask)[0]
    for yi in target_years:
        block_year = out[pix_idx, yi, :]
        band_std = np.nanstd(block_year, axis=0)
        band_std = np.where(np.isfinite(band_std), band_std, 0.0)
        sigma = sigma_ratio * np.maximum(band_std, 1e-8)
        noise = rng_local.normal(0.0, sigma, size=block_year.shape)
        out[pix_idx, yi, :] = block_year + noise

    return out


def _apply_method_lazy(base_da, method_name):
    arr = base_da.data
    year_mask = _year_mask_values(n_years, PERTURB_YEAR_INDICES)

    if method_name == "mean_variance_shift":
        out = da.map_blocks(
            _mean_shift_block,
            arr,
            year_mask=year_mask,
            band_index=RED_BAND_INDEX,
            factor=MEAN_SHIFT_FACTOR,
            add_constant=MEAN_SHIFT_ADD_CONSTANT,
            dtype=arr.dtype,
        )
    elif method_name == "covariance_shift":
        out = da.map_blocks(
            _cov_shift_block,
            arr,
            year_mask=year_mask,
            band_indices=np.array(COV_SHIFT_BAND_INDICES, dtype=int),
            strength=COV_SHIFT_STRENGTH,
            dtype=arr.dtype,
        )
    elif method_name == "noise_injection":
        out = da.map_blocks(
            _noise_block,
            arr,
            year_mask=year_mask,
            sigma_ratio=NOISE_SIGMA_RATIO,
            pixel_fraction=NOISE_PIXEL_FRACTION,
            random_state=RANDOM_STATE,
            dtype=arr.dtype,
        )
    else:
        raise ValueError(f"Unknown method: {method_name}")

    return xr.DataArray(
        out,
        dims=base_da.dims,
        coords=base_da.coords,
        attrs=base_da.attrs,
        name=base_da.name,
    )


def _save_ds_with_metadata(ds_synth, output_dir, artifact_stem, metadata):
    output_dir.mkdir(parents=True, exist_ok=True)
    zarr_path = output_dir / f"{artifact_stem}.zarr"
    json_path = output_dir / f"{artifact_stem}.metadata.json"

    if zarr_path.exists():
        shutil.rmtree(zarr_path)

    ds_synth.to_zarr(zarr_path, mode="w")

    with open(json_path, "w", encoding="utf-8") as handle:
        json.dump(metadata, handle, indent=2)

    return zarr_path, json_path

In [ ]:
# -----------------------------
# Run scalable generation (full dataset)
# -----------------------------
methods = ["mean_variance_shift", "covariance_shift", "noise_injection"]
artifact_rows = []

# Section A base: replicate baseline year to all years lazily
baseline_2d = s2_lazy.isel({year_dim: baseline_idx})
base_a = baseline_2d.expand_dims({year_dim: ds_source[year_dim]}).transpose(pixel_dim, year_dim, band_dim)

for method_name in methods:
    da_method = _apply_method_lazy(base_a, method_name)
    ds_synth = ds_source.copy(deep=False)
    ds_synth["s2_bands"] = da_method
    ds_synth = ds_synth.assign_coords(
        synthetic_year_label=(year_dim, np.array(synthetic_year_labels, dtype=object))
    )

    stem = (
        f"synthetic_mode-baseline_replication"
        f"_method-{method_name}"
        f"_baseline-{BASELINE_YEAR}"
        f"_perturbed-{years_suffix(perturb_year_values)}"
    )

    meta = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "source_dataset": DATA_PATH,
        "mode": "baseline_replication",
        "method": method_name,
        "baseline_year": BASELINE_YEAR,
        "perturb_year_indices": PERTURB_YEAR_INDICES,
        "perturb_year_values": perturb_year_values,
        "synthetic_year_labels": synthetic_year_labels,
        "config": RUN_CONFIG,
        "engine": "xarray+dask_chunked",
    }

    zarr_path, json_path = _save_ds_with_metadata(
        ds_synth,
        OUTPUT_ROOT / "section_a_baseline_replication",
        stem,
        meta,
    )

    artifact_rows.append(
        {
            "mode": "baseline_replication",
            "method": method_name,
            "baseline_year": BASELINE_YEAR,
            "perturb_year_indices": PERTURB_YEAR_INDICES,
            "perturb_year_values": perturb_year_values,
            "zarr_path": str(zarr_path),
            "metadata_path": str(json_path),
        }
    )
    print(f"Saved: {zarr_path}")

# Section B base: original per-year data lazily
base_b = s2_lazy
for method_name in methods:
    da_method = _apply_method_lazy(base_b, method_name)
    ds_synth = ds_source.copy(deep=False)
    ds_synth["s2_bands"] = da_method
    ds_synth = ds_synth.assign_coords(
        synthetic_year_label=(year_dim, np.array(synthetic_year_labels, dtype=object))
    )

    stem = (
        f"synthetic_mode-per_year_original"
        f"_method-{method_name}"
        f"_perturbed-{years_suffix(perturb_year_values)}"
    )

    meta = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "source_dataset": DATA_PATH,
        "mode": "per_year_original",
        "method": method_name,
        "baseline_year": None,
        "perturb_year_indices": PERTURB_YEAR_INDICES,
        "perturb_year_values": perturb_year_values,
        "synthetic_year_labels": synthetic_year_labels,
        "config": RUN_CONFIG,
        "engine": "xarray+dask_chunked",
    }

    zarr_path, json_path = _save_ds_with_metadata(
        ds_synth,
        OUTPUT_ROOT / "section_b_per_year_original",
        stem,
        meta,
    )

    artifact_rows.append(
        {
            "mode": "per_year_original",
            "method": method_name,
            "baseline_year": None,
            "perturb_year_indices": PERTURB_YEAR_INDICES,
            "perturb_year_values": perturb_year_values,
            "zarr_path": str(zarr_path),
            "metadata_path": str(json_path),
        }
    )
    print(f"Saved: {zarr_path}")

manifest_df = pd.DataFrame(artifact_rows).sort_values(["mode", "method"]).reset_index(drop=True)
manifest_csv = OUTPUT_ROOT / "synthetic_generation_manifest.csv"
manifest_json = OUTPUT_ROOT / "synthetic_generation_manifest.json"
manifest_df.to_csv(manifest_csv, index=False)

with open(manifest_json, "w", encoding="utf-8") as handle:
    json.dump(
        {
            "generated_at_utc": datetime.now(timezone.utc).isoformat(),
            "source_dataset": DATA_PATH,
            "baseline_year": BASELINE_YEAR,
            "perturb_year_indices": PERTURB_YEAR_INDICES,
            "perturb_year_values": perturb_year_values,
            "rows": manifest_df.to_dict(orient="records"),
        },
        handle,
        indent=2,
    )

print(f"Manifest CSV: {manifest_csv}")
print(f"Manifest JSON: {manifest_json}")
display(manifest_df)